# Aplicación de transcripción de audio con OpenAI Whisper

**Duración estimada:** 25 minutos

## Propósito
En este cuaderno vas a construir una interfaz gráfica sencilla que permita cargar archivos de audio y obtener su transcripción automática. Para esto, vas a integrar dos herramientas clave:
- **OpenAI Whisper**: un modelo de reconocimiento de voz (ASR) robusto y de código abierto.
- **Gradio**: una librería que permite crear interfaces de usuario para modelos de Machine Learning con muy pocas líneas de código.

Al finalizar, vas a tener una aplicación funcional que puede procesar audio en diversos idiomas y formatos, lista para usar en un entorno de laboratorio o investigación.

## Resultados de aprendizaje
Al terminar este cuaderno vas a poder:
- configurar un pipeline de reconocimiento de voz usando la librería `transformers`;
- definir funciones de procesamiento de audio en Python;
- construir y lanzar una interfaz web interactiva con Gradio.

---
## 1. Instalación de dependencias

Para que la aplicación funcione, necesitás instalar `gradio` (para la interfaz) y `transformers` junto con `torch` (para ejecutar el modelo Whisper).

In [ ]:
# Instalamos las librerías necesarias en modo silencioso
!pip install gradio transformers torch accelerate -q

---
## 2. Importación de librerías

Ahora cargamos los componentes necesarios. Vamos a usar el `pipeline` de Transformers, que simplifica enormemente la carga y el uso de modelos pre-entrenados.

In [ ]:
import torch
from transformers import pipeline
import gradio as gr

print("Librerías cargadas correctamente.")

---
## 3. Configuración de la lógica de transcripción

Vas a definir una función que reciba la ruta de un archivo de audio y devuelva el texto transcrito. El modelo elegido es `whisper-small`, que ofrece un buen equilibrio entre precisión y velocidad de ejecución.

In [ ]:
def transcribir_audio(archivo_audio):
    """
    Recibe un archivo de audio y devuelve su transcripción usando Whisper.
    """
    # Inicializamos el pipeline de reconocimiento de voz (ASR)
    # El parámetro chunk_length_s permite procesar audios largos dividiéndolos en fragmentos
    pipe = pipeline(
        "automatic-speech-recognition",
        model="openai/whisper-small",
        chunk_length_s=30,
        device="cuda" if torch.cuda.is_available() else "cpu"
    )

    # Procesamos el archivo y extraemos el texto
    resultado = pipe(archivo_audio, batch_size=8)["text"]
    return resultado

---
## 4. Construcción de la interfaz gráfica

Con Gradio, definís los componentes de entrada (`gr.Audio`) y salida (`gr.Textbox`). La clase `gr.Interface` se encarga de conectar esos componentes con la función que definiste anteriormente.

In [ ]:
# Definimos el componente de entrada: audio cargado desde la computadora
entrada_audio = gr.Audio(sources="upload", type="filepath", label="Cargar audio")

# Definimos el componente de salida: cuadro de texto para la transcripción
salida_texto = gr.Textbox(label="Transcripción resultante")

# Creamos la interfaz conectando lógica y componentes
demo = gr.Interface(
    fn=transcribir_audio,
    inputs=entrada_audio,
    outputs=salida_texto,
    title="Laboratorio NLP: Transcriptor con Whisper",
    description="Subí un archivo de audio para que el modelo identifique el idioma y genere la transcripción."
)

---
## 5. Lanzamiento de la aplicación

Al ejecutar la siguiente celda, vas a ver la interfaz directamente en el notebook. Si estás en Google Colab, se va a generar un enlace público temporal para que puedas probarla desde otros dispositivos.

In [ ]:
# Lanzamos la aplicación
demo.launch(inline=True)

---
## Análisis y reflexión

Considerá los siguientes puntos sobre el funcionamiento de la app:
- **Latencia**: ¿Cuánto tarda el modelo en procesar un minuto de audio? ¿Cómo afecta el uso de GPU o CPU?
- **Precisión**: ¿Cómo maneja el modelo los modismos locales o el ruido de fondo?
- **Escalabilidad**: ¿Qué cambios harías para que esta app soporte audios de varias horas de duración?

Este cuaderno demostró que podés pasar de un modelo complejo (Whisper) a una herramienta interactiva en cuestión de minutos. Esta capacidad es fundamental para prototipar soluciones de PLN de manera ágil.